# INFO 2950 Final Project Phase IV

## Introduction

### Research Questions
Question: Can we accurately predict the daily bike rental sharing count based on the environemntal conditions (weather, temp, hum, windspeed) and if that day is holiday or not.

In this analysis, we aim to explore correlations between daily bike rental counts and various factors, such as weather conditions and holidays. Specifically, we will investigate the relationships between temperature, humidity, wind speed, weather conditions and holiday. We will train a multivariable regression model to see if we can reliably predict the number of daily bike rentals based on these environmental and holiday conditions.

## Data Cleaning

We downloaded the data from UC Irvine Machine Learning Repository in the form of a .csv file. To perform data cleaning, we placed the .csv file in the same folder as our Phase 2, Our first steps are to load the dataset and check for null values.

In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import duckdb
import datetime

In [ ]:
# load data
data = pd.read_csv("hour.csv")
data.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.29,0.81,0.00,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.27,0.80,0.00,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.27,0.80,0.00,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.29,0.75,0.00,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.29,0.75,0.00,0,1,1


Since we are only interested in the total count of daily bike rentals, we are going to delete the causal and registered user count columns. 


In [ ]:
# remove "casual", "registered" column from the data
data = data.drop(data.columns[[-3, -2]], axis=1)
data.head()

,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.29,0.81,0.00,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.27,0.80,0.00,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.27,0.80,0.00,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.29,0.75,0.00,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.29,0.75,0.00,1


The description for the workingday column is “if the day is neither weekend nor holiday is 1, otherwise is 0.” Since the workingday column and the weekday and holiday columns are mostly based off of each other, we are going to delete the weekday and holiday columns to simplify the regression model. We do not want a regression model that is too complex and has high variance.

In [ ]:
# remove the holiday, weekend column
data = data.drop(['holiday', 'weekday'], axis=1)
data.head()

,instant,dteday,season,yr,mnth,hr,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,2011-01-01,1,0,1,0,0,1,0.24,0.29,0.81,0.00,16
1,2,2011-01-01,1,0,1,1,0,1,0.22,0.27,0.80,0.00,40
2,3,2011-01-01,1,0,1,2,0,1,0.22,0.27,0.80,0.00,32
3,4,2011-01-01,1,0,1,3,0,1,0.24,0.29,0.75,0.00,13
4,5,2011-01-01,1,0,1,4,0,1,0.24,0.29,0.75,0.00,1


We find that the dataset contains a column "instant" that works as an index column. We decided to remove this column.

In [ ]:
# drop the first column "instant"
data = data.drop(['instant'], axis=1)
data.head()

,dteday,season,yr,mnth,hr,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,2011-01-01,1,0,1,0,0,1,0.24,0.29,0.81,0.00,16
1,2011-01-01,1,0,1,1,0,1,0.22,0.27,0.80,0.00,40
2,2011-01-01,1,0,1,2,0,1,0.22,0.27,0.80,0.00,32
3,2011-01-01,1,0,1,3,0,1,0.24,0.29,0.75,0.00,13
4,2011-01-01,1,0,1,4,0,1,0.24,0.29,0.75,0.00,1


We convert the "dteday" column to become a date_time variable column.

In [ ]:
data['dteday'] = pd.to_datetime(data['dteday'])

We decided to delete the year column as this is repetitive information from dteday. There are only two years available in this dataframe (2011 and 2012).

In [ ]:
data = data.drop(['yr'], axis=1)
data.head()

,dteday,season,mnth,hr,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,2011-01-01,1,1,0,0,1,0.24,0.29,0.81,0.00,16
1,2011-01-01,1,1,1,0,1,0.22,0.27,0.80,0.00,40
2,2011-01-01,1,1,2,0,1,0.22,0.27,0.80,0.00,32
3,2011-01-01,1,1,3,0,1,0.24,0.29,0.75,0.00,13
4,2011-01-01,1,1,4,0,1,0.24,0.29,0.75,0.00,1


We decided to divide the "hour" column into four groups:
- first group: hour 0-5 (dawn, 1)
- second group: hour 6-11 (morning, 2)
- third group: hour 12-17 (afternoon, 3)
- fourth group: hour: 18-23 (night, 4)

We realized that the "weathersit" variable may differ among this period of time. However, we could not find the weather situaiton score that occured the most among this time period as there might be multiple situations that occured same amount of time. We decided to take the average of the weather situation then round it to the nearest whole value.

In [ ]:
# create a new nolumn called "time_period" that assigns the hours into four groups: 1, 2, 3, 4
data_mod = duckdb.sql("SELECT \
                      CASE \
                      WHEN hr BETWEEN 0 AND 5 THEN 1 \
                      WHEN hr BETWEEN 6 AND 11 THEN 2 \
                      WHEN hr BETWEEN 12 AND 17 THEN 3 \
                      WHEN hr BETWEEN 18 AND 23 THEN 4 \
                      END AS time_period, *\
                      FROM data").df()
data_mod.head(10)

,time_period,dteday,season,mnth,hr,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,2011-01-01,1,1,0,0,1,0.24,0.29,0.81,0.00,16
1,1,2011-01-01,1,1,1,0,1,0.22,0.27,0.80,0.00,40
2,1,2011-01-01,1,1,2,0,1,0.22,0.27,0.80,0.00,32
3,1,2011-01-01,1,1,3,0,1,0.24,0.29,0.75,0.00,13
4,1,2011-01-01,1,1,4,0,1,0.24,0.29,0.75,0.00,1
5,1,2011-01-01,1,1,5,0,2,0.24,0.26,0.75,0.09,1
6,2,2011-01-01,1,1,6,0,1,0.22,0.27,0.80,0.00,2
7,2,2011-01-01,1,1,7,0,1,0.20,0.26,0.86,0.00,3
8,2,2011-01-01,1,1,8,0,1,0.24,0.29,0.75,0.00,8
9,2,2011-01-01,1,1,9,0,1,0.32,0.35,0.76,0.00,14


In [ ]:
# groupby time_period
# find the average of "weathersit", "temp", "atemp", "hum", "windspeed" in the time period
# and find the sum of total bike rental count occured in this time period
data_clean = duckdb.sql("SELECT time_period, dteday, season, mnth, workingday, \
                        AVG(weathersit) AS weathersit_score, \
                        AVG(temp) AS temp, \
                        AVG(atemp) AS atemp, \
                        AVG(hum) AS hum, \
                        AVG(windspeed) AS windspeed, \
                        SUM(cnt) AS cnt \
                        FROM data_mod \
                        GROUP BY dteday, season, mnth, workingday, time_period \
                        ORDER BY dteday, time_period").df()
data_clean.head(10)

,time_period,dteday,season,mnth,workingday,weathersit_score,temp,atemp,hum,windspeed,cnt
0,1,2011-01-01,1,1,0,1.17,0.23,0.28,0.78,0.01,103.00
1,2,2011-01-01,1,1,0,1.00,0.29,0.32,0.79,0.09,119.00
2,3,2011-01-01,1,1,0,1.83,0.44,0.44,0.77,0.29,554.00
3,4,2011-01-01,1,1,0,2.33,0.42,0.42,0.89,0.25,209.00
4,1,2011-01-02,1,1,0,2.00,0.45,0.45,0.94,0.24,52.00
5,2,2011-01-02,1,1,0,2.33,0.39,0.39,0.75,0.24,154.00
6,3,2011-01-02,1,1,0,2.33,0.35,0.34,0.69,0.19,442.00
7,4,2011-01-02,1,1,0,1.17,0.28,0.26,0.44,0.32,153.00
8,1,2011-01-03,1,1,1,1.00,0.19,0.16,0.46,0.36,11.00
9,2,2011-01-03,1,1,1,1.00,0.16,0.14,0.46,0.31,431.00


In [ ]:
# round the weathersit to its closest whole number
# then convert it back to int
data_clean['weathersit_score'] = round(data_clean['weathersit_score'])
data_clean['weathersit_score'] = data_clean['weathersit_score'].astype(int)
data_clean.head(10)

,time_period,dteday,season,mnth,workingday,weathersit_score,temp,atemp,hum,windspeed,cnt
0,1,2011-01-01,1,1,0,1,0.23,0.28,0.78,0.01,103.00
1,2,2011-01-01,1,1,0,1,0.29,0.32,0.79,0.09,119.00
2,3,2011-01-01,1,1,0,2,0.44,0.44,0.77,0.29,554.00
3,4,2011-01-01,1,1,0,2,0.42,0.42,0.89,0.25,209.00
4,1,2011-01-02,1,1,0,2,0.45,0.45,0.94,0.24,52.00
5,2,2011-01-02,1,1,0,2,0.39,0.39,0.75,0.24,154.00
6,3,2011-01-02,1,1,0,2,0.35,0.34,0.69,0.19,442.00
7,4,2011-01-02,1,1,0,1,0.28,0.26,0.44,0.32,153.00
8,1,2011-01-03,1,1,1,1,0.19,0.16,0.46,0.36,11.00
9,2,2011-01-03,1,1,1,1,0.16,0.14,0.46,0.31,431.00


## Data Description

### What are the observations (rows) and the attributes (columns)?

The observations (rows) represent a particular time period each day from 2011 and 2012 in the Capital bikeshare system of bike-sharing data, where the weather, seasonal information, and the number of bike rentals is recorded.

The attributes (columns) includes:
- time_period: 
    - 1: Hour 0-5 (dawn)
    - 2: Hour 6-11 (morning)
    - 3: Hour 12-17 (afternoon)
    - 4: Hour: 18-23 (night)	
- dteday: date
- season: 
    - 1: Spring
    - 2: Summer
    - 3: Fall
    - 4: Winter
- mnth: month ( 1 to 12)
- workingday: if day is neither weekend nor holiday is 1, otherwise is 0.
- weathersit_score:
    - 1: Clear, Few clouds, Partly cloudy, Partly cloudy
    - 2: Mist + Cloudy, Mist + Broken clouds, Mist + Few clouds, Mist
    - 3: Light Snow, Light Rain + Thunderstorm + Scattered clouds, Light Rain + Scattered clouds
    - 4: Heavy Rain + Ice Pallets + Thunderstorm + Mist, Snow + Fog
- temp: Normalized temperature in Celsius. The values are divided to 41 (max)	
- atemp: Normalized feeling temperature in Celsius. The values are divided to 50 (max)
- hum: Normalized humidity. The values are divided to 100 (max)	
- windspeed: Normalized wind speed. The values are divided to 67 (max)	
- cnt: count of total rental bikes including both casual and registered


### Why was this dataset created?

Hadi Fanaee-T and Joao Gama created this dataset to study how the number of bike rentals are influenced by various factors, including time and environmental conditions. It was used to develop models to predict the demand of bike-sharing, which is useful for planning and improving current bike-sharing systems.

### Who funded the creation of the dataset?

The creation of the bike-sharing dataset was funded by these sources:
1. European Regional Development Fund through the COMPETE Program.
2. Portuguese Funds through the Portuguese Foundation for Science and Technology (FCT) within the project FCOMP — 01-0124-FEDER-022701.
3. European Commission through the project MAESTRA (Grant Number ICT-2013-612944).


### What processes might have influenced what data was observed and recorded and what was not?


Since the data is related to a 2-year usage log of Capital Bike Sharing (CBS) at Washington, D.C., USA , the types of data observed and recorded are mostly for operational purposes. For example, the number of bikes rented per day was recorded for tracking and usage. CBS also needed to consider user privacy, so information such as user description and live bicycle location were not recorded.

### What preprocessing was done, and how did the data come to be in the form that you are using?


The authors normalized values for temperature, humidity, and wind speed to help the model reduce variance between different environmental variables. Additionally, my team and I deleted some attributes (columns) that we feel would be correlated with other attributes. We also transformed the observations (rows) from every hour to every time period (every six hours) because our results would be more interpretable to more readers. When transforming the hour data, we also transformed the weather data to be averaged out between the aggregated hours. 

### If people are involved, were they aware of the data collection and if so, what purpose did they expect the data to be used for?


The bikeshare users were probably not explicitly aware, but had a slight idea, that certain data from them was being collected. We believe these riders agreed to the general collection of data when using CBS by a terms and conditions agreement, which people usually skip through and agree. Generally, users would likely expect the data to be anonymized and used for operational improvements purposes. 


### Where can your raw source data be found, if applicable? Provide a link to the raw data (hosted on Github, in a Cornell Google Drive or Cornell Box)


Not applicable. We found our dataset on UCI Machine Learning Repository. 


## Preregistration Statements

### Statement 1

### Statement 2
Hypothesis 2: Workingday status affects bike rental counts, with fewer rentals expected during non-workdays. 

Analysis: We will run a linear regression with workingday status as a dummy variable (1 for working day, 0 for non-working day) and daily bike rental count as the output variable. We will test whether the coefficient for the workingday dummy (B_workingday) is less than 0, indicating that fewer bikes are rented on working days.

## Data Analysis

## Evaluation of Significance

## Conclusions

## Limitations

## Acknowledgements and Bibliography